# Tables 2.5 and 2.6

This notebook produces the thinning-rate calculations for Tables 2.5 and 2.6 using `thicknesstimeseriesmeanfinal.csv`.

- **Table 2.5:** regional composite curves → 500-year grid → finite differences → centered 2.5 kyr moving average.
- **Table 2.6:** site-level posterior-mean curves on the native 2-year grid → finite differences → centered 2.5 kyr moving average.

All rates are reported in m kyr⁻¹.

In [1]:
import numpy as np
import pandas as pd

df = pd.read_csv("thicknesstimeseriesmeanfinal.csv")

print("Rows:", len(df))
print("Sites:", df["Site"].nunique())
print("Age range:", df["Age"].min(), "to", df["Age"].max(), "years")

Rows: 600060
Sites: 60
Age range: 0 to 20000 years


## Site assignments

In [7]:
# Regional assignments used for Table 2.5
regions = {
    "Southern TAMs": [56, 57, 59, 60, 61, 62, 64, 67],
    "Amundsen Sea": [161, 164, 166],
    "Weddell Sea": [116, 121, 122, 125, 128, 129, 142, 144, 145],
    "Marie Byrd Land": [148, 149, 151, 152],
    "Central TAMs": [33, 34, 37, 46, 47, 75, 78, 79, 80],
    "Northern TAMs": [27, 28, 30, 42, 45, 49, 50, 52, 54, 68, 70],
    "Queen Maud Land": [4, 7, 11, 15, 17, 21, 22, 24, 40],
    "Antarctic Peninsula": [85, 90, 91, 97, 111, 114],
}

# Site labels and basins used for Table 2.6
site_info = [
    ("Queen Maud Land", 4, "POC-4", "Prince Olav Coast"),
    ("Queen Maud Land", 7, "POC-7", "Prince Olav Coast"),
    ("Queen Maud Land", 11, "POC-11", "Prince Olav Coast"),
    ("Queen Maud Land", 15, "ICC-15", "Ingrid Christensen Coast"),
    ("Queen Maud Land", 17, "ICC-17", "Ingrid Christensen Coast"),
    ("Queen Maud Land", 21, "J-21", "Jutulstraumen"),
    ("Queen Maud Land", 22, "J-22", "Jutulstraumen"),
    ("Queen Maud Land", 24, "M-24", "Mawson Coast"),
    ("Queen Maud Land", 40, "FI-40", "Fisher"),

    ("Northern TAMs", 27, "CT-27", "Campbell-Tinker"),
    ("Northern TAMs", 28, "A-28", "David"),
    ("Northern TAMs", 30, "DA-30", "David"),
    ("Northern TAMs", 42, "IS-42", "Islands"),
    ("Northern TAMs", 45, "AV-45", "Aviator"),
    ("Northern TAMs", 49, "DV-49", "Dry Valleys"),
    ("Northern TAMs", 50, "DV-50", "Dry Valleys"),
    ("Northern TAMs", 52, "DV-52", "Dry Valleys"),
    ("Northern TAMs", 54, "DV-54", "Dry Valleys"),
    ("Northern TAMs", 68, "T-68", "Tucker"),
    ("Northern TAMs", 70, "T-70", "Tucker"),

    ("Central TAMs", 33, "B-33", "Byrd"),
    ("Central TAMs", 34, "B-34", "Byrd"),
    ("Central TAMs", 37, "SK-37", "Skelton"),
    ("Central TAMs", 46, "RE-46", "Ross East"),
    ("Central TAMs", 47, "RE-47", "Ross East"),
    ("Central TAMs", 75, "RE-75", "Ross East"),
    ("Central TAMs", 78, "RE-78", "Ross East"),
    ("Central TAMs", 79, "RE-79", "Ross East"),
    ("Central TAMs", 80, "RE-80", "Ross East"),

    ("Southern TAMs", 56, "LK-56", "Lennox-King"),
    ("Southern TAMs", 57, "LK-57", "Lennox-King"),
    ("Southern TAMs", 59, "SH-59", "Shackleton"),
    ("Southern TAMs", 60, "SC-60", "Scott"),
    ("Southern TAMs", 61, "SC-61", "Scott"),
    ("Southern TAMs", 62, "M-62", "Mercer"),
    ("Southern TAMs", 64, "M-64", "Mercer"),
    ("Southern TAMs", 67, "M-67", "Mercer"),

    ("Antarctic Peninsula", 85, "JP-85", "Jason Peninsula"),
    ("Antarctic Peninsula", 90, "EGL-90", "Eastern Graham Land"),
    ("Antarctic Peninsula", 91, "EGL-91", "Eastern Graham Land"),
    ("Antarctic Peninsula", 97, "EGL-97", "Eastern Graham Land"),
    ("Antarctic Peninsula", 111, "EGL-111", "Eastern Graham Land"),
    ("Antarctic Peninsula", 114, "DR-114", "Drygalski"),

    ("Weddell Sea", 116, "AC-116", "Academy"),
    ("Weddell Sea", 121, "AC-121", "Academy"),
    ("Weddell Sea", 122, "AC-122", "Academy"),
    ("Weddell Sea", 125, "R-125", "Rutford"),
    ("Weddell Sea", 128, "H-128", "Hercules"),
    ("Weddell Sea", 129, "H-129", "Hercules"),
    ("Weddell Sea", 142, "AC-142", "Academy"),
    ("Weddell Sea", 144, "FO-144", "Foundation"),
    ("Weddell Sea", 145, "IN-145", "Institute"),

    ("Marie Byrd Land", 148, "SU-148", "Sulzberger"),
    ("Marie Byrd Land", 149, "SU-149", "Sulzberger"),
    ("Marie Byrd Land", 151, "SU-151", "Sulzberger"),
    ("Marie Byrd Land", 152, "SU-152", "Sulzberger"),

    ("Amundsen Sea", 161, "WG-161", "Walgreen Coast"),
    ("Amundsen Sea", 164, "LV-164", "Lucchitta-Velasco"),
    ("Amundsen Sea", 166, "PI-166", "Pine Island"),
]

## Table 2.5 — Regional thinning rates

In [8]:
age_500 = np.arange(500, 20001, 500)
regional_rows = []

for region, site_ids in regions.items():

    # Mean regional thickness history
    composite = (
        df[df["Site"].isin(site_ids)]
        .groupby("Age")["Thickness"]
        .mean()
        .sort_index()
    )

    # 500-year grid
    thickness_500 = np.interp(
        age_500,
        composite.index.to_numpy(),
        composite.to_numpy()
    )

    # Consecutive 500-year thinning rates, m kyr^-1
    rate = np.diff(thickness_500) / 500 * 1000

    # Centered 2.5 kyr = 5-interval moving average
    smooth = (
        pd.Series(rate)
        .rolling(window=5, center=True, min_periods=1)
        .mean()
    )

    regional_rows.append({
        "Region": region,
        "N sites": len(site_ids),
        "Min (m kyr^-1)": smooth.min(),
        "Mean (m kyr^-1)": smooth.mean(),
        "Max (m kyr^-1)": smooth.max(),
        "N intervals": len(smooth)
    })

table25 = pd.DataFrame(regional_rows)
table25

,Region,N sites,Min (m kyr^-1),Mean (m kyr^-1),Max (m kyr^-1),N intervals
0,Southern TAMs,8,16.293537,47.532218,88.533215,39
1,Amundsen Sea,3,3.655770,35.635189,51.461142,39
2,Weddell Sea,9,12.433011,29.111265,66.061939,39
3,Marie Byrd Land,4,13.917555,26.675693,62.009706,39
4,Central TAMs,9,10.596386,22.447726,39.376604,39
5,Northern TAMs,11,5.647663,20.199585,46.849397,39
6,Queen Maud Land,9,6.081814,19.502954,43.601175,39
7,Antarctic Peninsula,6,8.751980,16.129181,32.015244,39


In [9]:
# Rounded version for the thesis table
table25_display = table25.copy()

for col in ["Min (m kyr^-1)", "Mean (m kyr^-1)", "Max (m kyr^-1)"]:
    table25_display[col] = table25_display[col].round(1)

table25_display

,Region,N sites,Min (m kyr^-1),Mean (m kyr^-1),Max (m kyr^-1),N intervals
0,Southern TAMs,8,16.3,47.5,88.5,39
1,Amundsen Sea,3,3.7,35.6,51.5,39
2,Weddell Sea,9,12.4,29.1,66.1,39
3,Marie Byrd Land,4,13.9,26.7,62.0,39
4,Central TAMs,9,10.6,22.4,39.4,39
5,Northern TAMs,11,5.6,20.2,46.8,39
6,Queen Maud Land,9,6.1,19.5,43.6,39
7,Antarctic Peninsula,6,8.8,16.1,32.0,39


## Table 2.6 — Site-level thinning rates

In [10]:
# 2.5 kyr / 2-year grid = about 1250 intervals.
# Use an odd window so the moving average is centered.
WINDOW = 1251

site_rows = []

for region, site_id, site_name, basin in site_info:

    site = df[df["Site"] == site_id].sort_values("Age")

    if site.empty:
        continue

    age = site["Age"].to_numpy()
    thickness = site["Thickness"].to_numpy()

    # Finite-difference thinning rate, m kyr^-1
    rate = np.diff(thickness) / np.diff(age) * 1000

    # Centered 2.5 kyr moving average
    smooth = (
        pd.Series(rate)
        .rolling(window=WINDOW, center=True, min_periods=1)
        .mean()
    )

    site_rows.append({
        "Region": region,
        "Site": site_name,
        "Basin": basin,
        "Min (m kyr^-1)": smooth.min(),
        "Max (m kyr^-1)": smooth.max()
    })

table26 = pd.DataFrame(site_rows)
table26

,Region,Site,Basin,Min (m kyr^-1),Max (m kyr^-1)
0,Queen Maud Land,POC-4,Prince Olav Coast,-0.204509,107.655890
1,Queen Maud Land,POC-7,Prince Olav Coast,4.835272,29.058840
2,Queen Maud Land,POC-11,Prince Olav Coast,-0.087041,122.862414
3,Queen Maud Land,ICC-15,Ingrid Christensen Coast,0.004990,102.638313
4,Queen Maud Land,ICC-17,Ingrid Christensen Coast,0.221106,51.753343
5,Queen Maud Land,J-21,Jutulstraumen,-0.119736,87.755839
6,Queen Maud Land,J-22,Jutulstraumen,0.467887,38.283630
7,Queen Maud Land,M-24,Mawson Coast,-0.027374,109.703378
8,Queen Maud Land,FI-40,Fisher,0.263327,134.696063
9,Northern TAMs,CT-27,Campbell-Tinker,0.010853,10.230280


In [ ]:
# Rounded version for the thesis table
table26_display = table26.copy()

table26_display["Min (m kyr^-1)"] = table26_display["Min (m kyr^-1)"].round(2)
table26_display["Max (m kyr^-1)"] = table26_display["Max (m kyr^-1)"].round(1)

table26_display